# ReXKG Pipeline (PyHealth Style)

This notebook shows a PyHealth-native ReXKG workflow similar to other examples:

1. Load dataset
2. Set tasks
3. Build sample datasets
source

predictions_path = Path.cwd() / "result" / "run_relation" / "predictions.json"
if not predictions_path.exists():
    raise FileNotFoundError(f"Missing relation predictions file: {predictions_path}")

structured_output_path = Path.cwd() / "data" / "your_test_file.json"
processed_docs = rexkg_reverse_structure(
    input_json_file=str(predictions_path),
    save_json_file=str(structured_output_path),
)

print("Input predictions:", predictions_path)
print("Structured output:", structured_output_path)
print("Converted documents:", len(processed_docs))


## 1) Install needed libraries and import RexKG pyhealth implementation

In [1]:
# Make sure to install the needed libraries used for the rexkg PyHealth files. 
# Kernal for this conda virtual environment is running 3.13.13
#!python -m pip install neraug
#!python -m pip install torch

In [2]:
# may take a few mins to run to build cache
from pathlib import Path
import sys
import importlib.util


# Make sure the local PyHealth package is importable from this notebook.
# Notebook location: PyHealth/examples/rexkg/load_dataset.ipynb
# Package root:      PyHealth/
pyhealth_root = Path.cwd().resolve().parents[1]
if str(pyhealth_root) not in sys.path:
    sys.path.insert(0, str(pyhealth_root))

from pyhealth.datasets import RexKGDataset, RexKGCheXpertDataset 
# from pyhealth.models import RexKG
from pyhealth.tasks import (
    RexKGEntityExtractionRadiology,
    RexKGRelationExtractionRadiology,
    RexKGReverseStructureRadiology,
    RexKGGetEntitiesRadiology,
)

/home/strawhat/miniconda3/envs/cs598-pyhealth/lib/python3.13/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


## 2) Configure Paths

Set `PROJECT_ROOT` to your repo root if auto-detection does not match your environment.

In [3]:
PROJECT_ROOT = Path.cwd()
if not (PROJECT_ROOT / "PyHealth").exists():
    PROJECT_ROOT = PROJECT_ROOT.parent

print("PROJECT_ROOT:", PROJECT_ROOT)

PROJECT_ROOT: /mnt/c/Users/Aarje/OneDrive - University of Illinois - Urbana/School/cs598_project/PyHealth/examples
REXKG_DATA_ROOT: /mnt/c/Users/Aarje/OneDrive - University of Illinois - Urbana/School/cs598_project/PyHealth/examples/src/ner/data
Data root exists: False


## 3) Declare the dataset CheXpert Dataset
The CheXpert dataset `df_chexpert_plus_240401.csv` can be downloaded at:
https://stanfordaimi.azurewebsites.net/datasets/5158c524-d3ab-4e02-96e9-6ee9efc110a1

You may have to create and account and accept Terms of Agreement to access. 

In [16]:
cheXpert = RexKGCheXpertDataset(
    root=str(PROJECT_ROOT / "data" / "df_chexpert_plus_240401.csv"),
    table=[
        "path_to_image",
        "path_to_dcm",
        "frontal_lateral",
        "ap_pa",
        "deid_patient_id",
        "patient_report_date_order",
        "report",
        "section_narrative",
        "section_clinical_history",
        "section_history",
        "section_comparison",
        "section_technique",
        "section_procedure_comments",
        "section_findings",
        "section_impression",
        "section_end_of_impression",
        "section_summary",
        "section_accession_number",
        "age",
        "sex",
        "race",
        "ethnicity",
        "interpreter_needed",
        "insurance_type",
        "recent_bmi",
        "deceased",
        "split",
    ],
    dev=False
)

NameError: name 'RexKGCheXpertDataset' is not defined

# 3.5) Use Chat GPT 4 to label the entities and relations extraction

An **Entity** in the schema are categorized into six types as listed.
1. Anatomy: anatomical structures within the body.
2. Disorder: any abnormal findings or diseases identified within radiology reports.
3. Concept: descriptors used to modify other entities, for example, ”acute”, ”severe”, and ”increasing”.
4. Device: any instrument or apparatus used for medical purposes, for example, “tube”, “clip”, “wire”.
5. Procedure: medical procedures used to diagnose, measure, monitor, or treat conditions, such as “sternotomy”.
6. Size: measurements of disorders or anatomical structures, for example, “3-mm”.

A Relation is defined as a directed edge between two entities. Following the previous work (Jain et al. 2021a), our schema uses three relations as listed.
1. Suggestive of: source entity (e.g., findings) may suggest the presence of the target entity (e.g., a disease).
2. Located at: source entity is located at the target entity.
3. Modify: source entity modifies or provides additional information about the target entity.

RexKGGPT4EntityExtractionRadiology()

create a new task RexKGGPT4EntityExtractionRadiology that copies the exact way it was implemented in PyHealth. RexKGGPT4EntityExtractionRadiology should also take in and accept 

## 3.75) Load RexKGDataset - Data Preperation

In [4]:
# Prefer repo-relative split files created by src/ner/data/structure_data.py
split_root = PROJECT_ROOT / "PyHealth" / "examples" / "rexkg" / "data" / "data_split"
if not split_root.exists():
    # Fallback when running from PyHealth/examples/rexkg
    split_root = Path.cwd() / "data" / "data_split"

train_json = split_root / "train.json"
test_json = split_root / "test.json"
dev_json = split_root / "test.json"

for p in [train_json, dev_json, test_json]:
    if not p.exists():
        raise FileNotFoundError(f"Missing split file: {p}")

# dataset = RexKGDataset(root=str(expected))
train_dataset = RexKGDataset(root=str(train_json))
dev_dataset = RexKGDataset(root=str(dev_json))
test_dataset = RexKGDataset(root=str(test_json))

## 4) Apply ReXKG Pipeline Tasks 
should this be a model?

In [5]:
entity_task = RexKGEntityExtractionRadiology()


# Force output under this notebook folder.
entity_output_dir = Path.cwd() / "result" / "run_entity"
model = RexKGEntityExtractionRadiology.set_task(
    train_data=train_dataset,
    dev_data=dev_dataset,
    test_data=test_dataset,
    model="bert-base-uncased",
    output_dir=str(entity_output_dir),
    do_train=True,
    do_eval=True,
    eval_test=True,
    learning_rate=1e-5,
    task_learning_rate=5e-4,
    train_batch_size=8,
    eval_batch_size=64,
    num_epoch=1,
    context_window=5,
)
print(model)

pred_file = entity_output_dir / "ent_pred_mimic_headct.json"
print("Expected prediction file:", pred_file)
print("Prediction file exists:", pred_file.exists())

# I think I need to make these set_tasts above instead of run_entity_pipeline
# entity_samples = dataset.set_task(entity_task)





# relation_samples = dataset.set_task(relation_task)
# kg_samples = dataset.set_task(kg_task)

# print("Entity samples:", len(entity_samples))
# print("Relation samples:", len(relation_samples))
# print("KG samples:", len(kg_samples))

Some weights of BertForEntity were not initialized from the model checkpoint at bert-base-uncased and are newly initialized: ['ner_classifier.0.network.0.bias', 'ner_classifier.0.network.0.weight', 'ner_classifier.0.network.3.bias', 'ner_classifier.0.network.3.weight', 'ner_classifier.1.bias', 'ner_classifier.1.weight', 'width_embedding.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.
  0%|          | 0/1 [00:00<?, ?it/s]

Evaluating...
Accuracy: 0.973324
Cor: 2699, Pred TOT: 3632, Gold TOT: 3699
P: 0.74312, R: 0.72966, F1: 0.73633
Used time: 3.601631
Saving model to /mnt/c/Users/Aarje/OneDrive - University of Illinois - Urbana/School/cs598_project/PyHealth/examples/rexkg/result/run_entity...


100%|██████████| 1/1 [00:49<00:00, 49.35s/it]


Evaluating...
Accuracy: 0.973324
Cor: 2699, Pred TOT: 3632, Gold TOT: 3699
P: 0.74312, R: 0.72966, F1: 0.73633
Used time: 3.527679
Total pred entities: 3632
Output predictions to /mnt/c/Users/Aarje/OneDrive - University of Illinois - Urbana/School/cs598_project/PyHealth/examples/rexkg/result/run_entity/ent_pred_mimic_headct.json..
{'output_dir': '/mnt/c/Users/Aarje/OneDrive - University of Illinois - Urbana/School/cs598_project/PyHealth/examples/rexkg/result/run_entity', 'log_file': '/mnt/c/Users/Aarje/OneDrive - University of Illinois - Urbana/School/cs598_project/PyHealth/examples/rexkg/result/run_entity/train.log', 'task': 'mimic01', 'model': 'bert-base-uncased', 'best_dev_f1': 0.7363251943800299, 'test_f1': 0.7363251943800299, 'test_prediction_file': '/mnt/c/Users/Aarje/OneDrive - University of Illinois - Urbana/School/cs598_project/PyHealth/examples/rexkg/result/run_entity/ent_pred_mimic_headct.json'}
Expected prediction file: /mnt/c/Users/Aarje/OneDrive - University of Illinois -

## 5) Run Relation Pipeline

All input/output paths below stay inside `PyHealth/examples/rexkg/`.

should this be a model?

In [6]:
relation_output_dir = Path.cwd() / "result" / "run_relation_pyhealth_v2"

# ner_src_dir points to src/ner so BertForRelation and generate_relation_data can be imported.
# Auto-detection walks up from relation_output_dir; set explicitly if it fails.
NER_SRC_DIR = str(PROJECT_ROOT / "src" / "ner")

relation_metrics = RexKGRelationExtractionRadiology.set_task(
    train_file=str(train_json),
    entity_output_dir=str(entity_output_dir),
    entity_predictions_dev="ent_pred_mimic_headct.json",
    entity_predictions_test="ent_pred_mimic_headct.json",
    model="bert-base-uncased",
    output_dir=str(relation_output_dir),
    do_train=True,
    do_eval=True,
    eval_with_gold=True,
    do_lower_case=True,
    train_batch_size=16,
    eval_batch_size=32,
    learning_rate=5e-5,
    num_train_epochs=1,
    context_window=20,
    max_seq_length=256,
    ner_src_dir=NER_SRC_DIR,
)
print(relation_metrics)

relation_pred_file = relation_output_dir / "predictions.json"
print("Expected relation prediction file:", relation_pred_file)
print("Relation prediction file exists:", relation_pred_file.exists())

device: cuda, n_gpu: 1
Writing example 0 of 31904
*** Example ***
guid: 0@0::(0,0)-(1,1)
tokens: [CLS] [unused1] unchanged [unused2] [unused3] position [unused4] of the left upper ex ##tre ##mity pic ##c line . [SEP]
input_ids: 101 2 15704 3 4 2597 5 1997 1996 2187 3356 4654 7913 16383 27263 2278 2240 1012 102 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0
input_mask: 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 

Some weights of BertForRelation were not initialized from the model checkpoint at bert-base-uncased and are newly initialized: ['classifier.bias', 'classifier.weight', 'layer_norm.bias', 'layer_norm.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


Start epoch #0 (lr = 5e-05)...
Epoch: 0, Step: 19668 / 19668, used_time = 6309.74s, loss = 0.140988
***** Eval results *****
  accuracy = 0.9525764794383149
  eval_loss = 0.1309519582707108
  f1 = 0.6686507936507937
  n_correct = 1685
  n_gold = 2704
  n_pred = 2336
  precision = 0.721318493150685
  recall = 0.6231508875739645
  task_f1 = 0.6749449228920489
  task_ngold = 2657
  task_recall = 0.6341738803161461
!!! Best dev f1 (lr=5e-05, epoch=0): 66.87
Saving model to /mnt/c/Users/Aarje/OneDrive - University of Illinois - Urbana/School/cs598_project/PyHealth/examples/rexkg/result/run_relation_pyhealth_v2
Epoch: 0, Step: 19668 / 19668, used_time = 6515.31s, loss = 0.140988
***** Eval results *****
  accuracy = 0.9525764794383149
  eval_loss = 0.1309519582707108
  f1 = 0.6686507936507937
  n_correct = 1685
  n_gold = 2704
  n_pred = 2336
  precision = 0.721318493150685
  recall = 0.6231508875739645
  task_f1 = 0.6749449228920489
  task_ngold = 2657
  task_recall = 0.6341738803161461
{'S

# 6) Reverse Graph Constructed
Build entity/relation tables from reversed JSON

converts the relation‑extraction outputs back into the structured, report‑level format that the KG construction pipeline expects — i.e., it reverses the preprocessing that turned raw reports into PURE training/test examples so the predicted entities/relations can be used for node and edge construction. 

In [7]:
predictions_path = Path.cwd() / "result" / "run_relation" / "predictions.json"
if not predictions_path.exists():
    raise FileNotFoundError(f"Missing relation predictions file: {predictions_path}")

structured_output_path = Path.cwd() / "data" / "your_test_file.json"
processed_docs = RexKGReverseStructureRadiology.set_task(
    input_json_file=str(predictions_path),
    save_json_file=str(structured_output_path),
)

print("Input predictions:", predictions_path)
print("Structured output:", structured_output_path)
print("Converted documents:", len(processed_docs))

print("\nFirst 10 lines of structured output JSON:")
with structured_output_path.open("r", encoding="utf-8") as f:
    for i, line in enumerate(f, start=1):
        if 2231 <= i <= 2282:
            print(f"{i:04d}: {line.rstrip()}")
        if i > 2282:
            break

Input predictions: /mnt/c/Users/Aarje/OneDrive - University of Illinois - Urbana/School/cs598_project/PyHealth/examples/rexkg/result/run_relation/predictions.json
Structured output: /mnt/c/Users/Aarje/OneDrive - University of Illinois - Urbana/School/cs598_project/PyHealth/examples/rexkg/data/your_test_file.json
Converted documents: 560

First 10 lines of structured output JSON:
2231:     {
2232:         "doc_key": "68",
2233:         "sentences": "compared to prior chest x-ray on february 15th , pa and lateral upright views of the chest again demonstrate postsurgical changes of the mitral valve replacement and tricuspid valve anuloplasty .",
2234:         "entities": {
2235:             "prior": "concept",
2236:             "chest": "anatomy",
2237:             "x-ray": "disorder_present",
2238:             "pa": "anatomy",
2239:             "lateral": "concept",
2240:             "upright": "concept",
2241:             "views": "concept",
2242:             "postsurgical": "concept",


## 7) Get Entities 
data task --- will local smoke be fine???

In [8]:
# Build entity and relation summary CSV files from Step 5 output JSON.
entity_csv_output_dir = Path.cwd() / "result" / "local_run" / "entities"
relation_csv_output_dir = Path.cwd() / "result" / "local_run" / "relation"

get_entities_result = RexKGGetEntitiesRadiology.set_task(
    ent_pred_mimic_headct=str(structured_output_path),
    ent_real_pred_mimic_headct=str(structured_output_path),
    save_entity_dir=str(entity_csv_output_dir),
    save_real_dir=str(relation_csv_output_dir),
)

print(get_entities_result)
print("Entity CSV folder:", entity_csv_output_dir)
print("Relation CSV folder:", relation_csv_output_dir)
print("all_entities.csv exists:", (entity_csv_output_dir / "all_entities.csv").exists())
print("all_relations.csv exists:", (relation_csv_output_dir / "all_relations.csv").exists())

100%|██████████| 560/560 [00:00<00:00, 715010.73it/s]

{'all_entities_csv': '/mnt/c/Users/Aarje/OneDrive - University of Illinois - Urbana/School/cs598_project/PyHealth/examples/rexkg/result/local_run/entities/all_entities.csv', 'all_relations_csv': '/mnt/c/Users/Aarje/OneDrive - University of Illinois - Urbana/School/cs598_project/PyHealth/examples/rexkg/result/local_run/relation/all_relations.csv', 'save_entity_dir': '/mnt/c/Users/Aarje/OneDrive - University of Illinois - Urbana/School/cs598_project/PyHealth/examples/rexkg/result/local_run/entities', 'save_real_dir': '/mnt/c/Users/Aarje/OneDrive - University of Illinois - Urbana/School/cs598_project/PyHealth/examples/rexkg/result/local_run/relation'}
Entity CSV folder: /mnt/c/Users/Aarje/OneDrive - University of Illinois - Urbana/School/cs598_project/PyHealth/examples/rexkg/result/local_run/entities
Relation CSV folder: /mnt/c/Users/Aarje/OneDrive - University of Illinois - Urbana/School/cs598_project/PyHealth/examples/rexkg/result/local_run/relation
all_entities.csv exists: True
all_rel

## 8) Get UMLS 
models

In [9]:
# !python get_umls_entities.py --save_entity_dir ../result/local_run/entities

## 9) Filter CUI

In [10]:
# !python filter_cui.py --save_entity_dir ../result/local_run/entities

## 10) Structure Entities

In [11]:
# !python structure_entities.py --save_entity_dir ../result/local_run/entities --ignore_count 1



## 11) get kg nodes

In [12]:
# !python get_kg_nodes.py \
#   --save_entity_dir ../result/local_run/entities \
#   --save_real_dir ../result/local_run/relation \
#   --save_kg_dir ../result/local_run/kg



## 12) get size of relations

In [13]:
# !python get_size_relations.py \
#   --entity_dir ../result/local_run/entities \
#   --real_dir ../result/local_run/relation


## 13) get inference

this should be in metrics

In [14]:

# python get_inference_data.py

# !ls -lh /content/drive/MyDrive/cs598_project/src/kg_construct/result/local_run/kg